In [2]:
import os
from pathlib import Path

In [3]:
# --- Basic local project config ---
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)


In [4]:
!pip install -q requests tqdm

In [5]:
import json
import pandas as pd
import re
import time
import requests
from tqdm.notebook import tqdm
from collections import Counter

# Data Fetching

In [6]:
class UniProtDataFetcher:
    """Fetches protein data from UniProt REST API with robust pagination"""

    BASE_URL = "https://rest.uniprot.org/uniprotkb/search"

    def __init__(self, max_proteins=10000):
        self.max_proteins = max_proteins
        self.proteins = []
        self.accession_ids = set()  # Track unique accession IDs
        self.duplicate_count = 0

    def fetch_proteins(self, query="reviewed:true AND organism_id:9606"):
        """
        Fetch high-quality proteins from UniProt with robust pagination

        Query filters:
        - reviewed:true      = Swiss-Prot (manually reviewed)
        - organism_id:9606   = Human
        """

        print(f"🔍 Fetching proteins from UniProt...")
        print(f"📋 Query: {query}")
        print(f"🎯 Target: {self.max_proteins} proteins\n")

        params = {
            "query": query,
            "format": "json",
            "size": 500,  # Maximum batch size allowed by API
            "fields": (
                "accession,id,protein_name,cc_function,sequence,length,"
                "organism_name,gene_names,go_id,ec,keyword"
            ),
        }

        # Cursor in Link header, not JSON response
        cursor = None
        retry_count = 0
        max_retries = 3

        with tqdm(total=self.max_proteins, desc="Downloading", unit="proteins") as pbar:
            while len(self.proteins) < self.max_proteins:
                # Use cursor parameter if available
                if cursor:
                    params["cursor"] = cursor
                elif "cursor" in params:
                    # Remove cursor param for first request
                    del params["cursor"]

                try:
                    response = requests.get(self.BASE_URL, params=params, timeout=30)
                    response.raise_for_status()
                    data = response.json()

                    # Check if we have results
                    if "results" not in data or len(data["results"]) == 0:
                        print("\n✓ No more results available from UniProt")
                        break

                    # Process this batch
                    batch_added = 0
                    batch_skipped = 0
                    for entry in data["results"]:
                        if len(self.proteins) >= self.max_proteins:
                            break

                        protein_data = self._parse_entry(entry)
                        if protein_data:
                            # Check for duplicate accession IDs
                            acc_id = protein_data["accession"]
                            if acc_id in self.accession_ids:
                                self.duplicate_count += 1
                                continue

                            self.accession_ids.add(acc_id)
                            self.proteins.append(protein_data)
                            batch_added += 1
                            pbar.update(1)
                        else:
                            batch_skipped += 1

                    # Log batch statistics every 5000 proteins
                    if len(self.proteins) % 5000 == 0 and len(self.proteins) > 0:
                        print(f"\n  Progress: {len(self.proteins)} proteins collected")

                    # Extract cursor from Link header
                    cursor = self._extract_cursor_from_headers(response.headers)

                    # If no cursor in headers, we've reached the end
                    if not cursor:
                        print("\n✓ Reached end of available results (no Link header)")
                        break

                    # Reset retry count on success
                    retry_count = 0

                    time.sleep(0.3)

                except requests.RequestException as e:
                    retry_count += 1
                    if retry_count >= max_retries:
                        print(f"\n❌ Failed after {max_retries} retries: {e}")
                        break

                    print(f"\n⚠️  Error (retry {retry_count}/{max_retries}): {e}")
                    time.sleep(2)  # Wait before retry

        print(f"\n✓ Successfully fetched {len(self.proteins)} proteins")
        print(f"  - Unique accession IDs: {len(self.accession_ids)}")
        print(f"  - Duplicates found: {self.duplicate_count}")

        return self.proteins

    def _extract_cursor_from_headers(self, headers):
        """
        Extract cursor from Link header.

        UniProt returns cursor in Link header like:
        Link: <https://rest.uniprot.org/...?cursor=ABCD1234>; rel="next"
        """
        link_header = headers.get("Link", "")
        if not link_header:
            return None

        # Parse Link header to extract cursor
        # Format: <url?cursor=VALUE>; rel="next"       
        match = re.search(r'cursor=([^&>]+)', link_header)
        if match:
            return match.group(1)

        return None

    def _parse_entry(self, entry):
        """Parse UniProt entry into structured format"""
        try:
            # Accession ID (primary unique identifier)
            accession = entry.get("primaryAccession", "")

            # Protein name
            protein_name = (
                entry.get("proteinDescription", {})
                .get("recommendedName", {})
                .get("fullName", {})
                .get("value", "")
            )

            # Sequence
            sequence = entry.get("sequence", {}).get("value", "")
            seq_length = entry.get("sequence", {}).get("length", 0)

            # Organism
            organism = entry.get("organism", {}).get("scientificName", "")

            # Gene name
            genes = entry.get("genes", [])
            gene_name = genes[0].get("geneName", {}).get("value", "") if genes else ""

            # Function annotation from comments
            function = ""
            comments = entry.get("comments", [])
            for comment in comments:
                if comment.get("commentType") == "FUNCTION":
                    for text in comment.get("texts", []):
                        function += text.get("value", "") + " "
            function = function.strip()

            # Quality filters
            if not sequence:  # Must have sequence
                return None
            if not function:  # Must have function annotation
                return None
            if seq_length < 50:  # Very short sequences might be fragments
                return None

            # Clean function text
            function = self._clean_text(function)

            # After cleaning, ensure we still have something meaningful
            if len(function) < 20:  # Reduced from 50
                return None

            # GO terms extraction
            go_terms = []
            for xref in entry.get("uniProtKBCrossReferences", []):
                if xref.get("database") == "GO":
                    go_id = xref.get("id")
                    go_term = None
                    for prop in xref.get("properties", []):
                        if prop.get("key") == "GoTerm":
                            go_term = prop.get("value")
                            break
                    go_terms.append({
                        "go_id": go_id,
                        "go_term": go_term
                    })

            # EC numbers extraction
            ec_numbers = []
            ec_list = (
                entry.get("proteinDescription", {})
                    .get("recommendedName", {})
                    .get("ecNumbers", [])
            )
            for ec in ec_list:
                value = ec.get("value")
                if value:
                    ec_numbers.append(value)
                    
            # Keywords extraction
            keywords = []
            for keyword in entry.get("keywords", []):
                keyword_name = keyword.get("name") or keyword.get("value")
                if keyword_name:
                    keywords.append({
                        "id": keyword.get("id"),
                        "name": keyword_name,
                        "category": keyword.get("category"),
                    })

            return {
                "accession": accession,
                "protein_name": protein_name,
                "gene_name": gene_name,
                "organism": organism,
                "sequence": sequence,
                "length": seq_length,
                "function": function,
                "go_terms": go_terms,
                "ec_numbers": ec_numbers,
                "keywords": keywords,
            }

        except Exception as e:
            # Skip malformed entries silently
            return None

    def _clean_text(self, text: str) -> str:
        """Clean and normalize UniProt function text"""
        # Remove references like {ECO:0000269|PubMed:12345}
        text = re.sub(r"\{[^}]+\}", " ", text)

        # Normalize spaces
        text = re.sub(r"\s+", " ", text).strip()

        # Keep ALL sentences instead of truncating to 3
        # (We want complete function descriptions for your MLLM training)

        # Remove weird characters but preserve basic punctuation
        text = re.sub(r"[^\w\s\.,;:()\-/']", "", text)
        text = re.sub(r"\s+", " ", text).strip()

        return text

    def save_raw_data(self, filepath):
        """Save raw protein data to JSON"""
        with open(filepath, "w") as f:
            json.dump(self.proteins, f, indent=2)
        print(f"✓ Saved raw data to {filepath}")

    def verify_uniqueness(self):
        """Verify that accession IDs are unique"""
        accessions = [p["accession"] for p in self.proteins]
        counter = Counter(accessions)
        duplicates = {k: v for k, v in counter.items() if v > 1}

        print(f"\n🔍 Uniqueness Verification:")
        print(f"  - Total proteins: {len(self.proteins)}")
        print(f"  - Unique accession IDs: {len(set(accessions))}")
        print(f"  - Duplicates: {len(duplicates)}")

        if duplicates:
            print(f"  ⚠️  Found duplicate accession IDs:")
            for acc, count in list(duplicates.items())[:5]:
                print(f"    - {acc}: {count} times")
        else:
            print(f"  ✓ All accession IDs are unique!")

        return len(duplicates) == 0

In [7]:
MAX_PROTEINS = 10000

fetcher = UniProtDataFetcher(max_proteins=MAX_PROTEINS)

# Fetch proteins with cursor-based pagination
proteins = fetcher.fetch_proteins(
    query="reviewed:true AND organism_id:9606"
)

🔍 Fetching proteins from UniProt...
📋 Query: reviewed:true AND organism_id:9606
🎯 Target: 10000 proteins



Downloading:   0%|          | 0/10000 [00:00<?, ?proteins/s]


  Progress: 10000 proteins collected

✓ Successfully fetched 10000 proteins
  - Unique accession IDs: 10000
  - Duplicates found: 0


In [8]:
#Verify Accession ID Uniqueness

is_unique = fetcher.verify_uniqueness()


🔍 Uniqueness Verification:
  - Total proteins: 10000
  - Unique accession IDs: 10000
  - Duplicates: 0
  ✓ All accession IDs are unique!


In [9]:
raw_file = RAW_DIR / "proteins_raw_10k.json"
fetcher.save_raw_data(raw_file)

✓ Saved raw data to C:\Users\sridi\Documents\Learning\protein-function-active-learning\protein-function-active-learning\data\raw\proteins_raw_10k.json


In [10]:
if proteins:
    print(f"\nDataset Statistics:")
    print(f"  - Total proteins: {len(proteins)}")
    print(f"  - Average sequence length: {sum(p['length'] for p in proteins) / len(proteins):.0f}")
    print(f"  - Average function length (chars): {sum(len(p['function']) for p in proteins) / len(proteins):.0f}")
    print(f"  - Proteins with GO terms: {sum(1 for p in proteins if p['go_terms'])}")
    print(f"  - Proteins with EC numbers: {sum(1 for p in proteins if p['ec_numbers'])}")



Dataset Statistics:
  - Total proteins: 10000
  - Average sequence length: 624
  - Average function length (chars): 674
  - Proteins with GO terms: 9994
  - Proteins with EC numbers: 2796


In [11]:
# Show sample protein
print(f"\nSample Protein:")
sample = proteins[0]
print(f"  - Accession: {sample['accession']}")
print(f"  - Name: {sample['protein_name']}")
print(f"  - Gene: {sample['gene_name']}")
print(f"  - Organism: {sample['organism']}")
print(f"  - Sequence: {sample['sequence']}")
print(f"  - Sequence Length: {sample['length']}")
print(f"  - Function Description: {sample['function']}")
print(f"  - GO terms: {len(sample['go_terms'])}")
print(f"  - EC numbers: {sample['ec_numbers']}")
print(f"  - Keywords: {sample['keywords']}")



Sample Protein:
  - Accession: A0A1B0GTW7
  - Name: Ciliated left-right organizer metallopeptidase
  - Gene: CIROP
  - Organism: Homo sapiens
  - Sequence: MLLLLLLLLLLPPLVLRVAASRCLHDETQKSVSLLRPPFSQLPSKSRSSSLTLPSSRDPQPLRIQSCYLGDHISDGAWDPEGEGMRGGSRALAAVREATQRIQAVLAVQGPLLLSRDPAQYCHAVWGDPDSPNYHRCSLLNPGYKGESCLGAKIPDTHLRGYALWPEQGPPQLVQPDGPGVQNTDFLLYVRVAHTSKCHQETVSLCCPGWSTAAQSQLTAALTSWAQRRGFVMLPRLCLKLLGSSNLPTLASQSIRITGPSVIAYAACCQLDSEDRPLAGTIVYCAQHLTSPSLSHSDIVMATLHELLHALGFSGQLFKKWRDCPSGFSVRENCSTRQLVTRQDEWGQLLLTTPAVSLSLAKHLGVSGASLGVPLEEEEGLLSSHWEARLLQGSLMTATFDGAQRTRLDPITLAAFKDSGWYQVNHSAAEELLWGQGSGPEFGLVTTCGTGSSDFFCTGSGLGCHYLHLDKGSCSSDPMLEGCRMYKPLANGSECWKKENGFPAGVDNPHGEIYHPQSRCFFANLTSQLLPGDKPRHPSLTPHLKEAELMGRCYLHQCTGRGAYKVQVEGSPWVPCLPGKVIQIPGYYGLLFCPRGRLCQTNEDINAVTSPPVSLSTPDPLFQLSLELAGPPGHSLGKEQQEGLAEAVLEALASKGGTGRCYFHGPSITTSLVFTVHMWKSPGCQGPSVATLHKALTLTLQKKPLEVYHGGANFTTQPSKLLVTSDHNPSMTHLRLSMGLCLMLLILVGVMGTTAYQKRATLPVRPSASYHSPELHSTRVPVRGIREV
  - Sequence Length: 788
  - Function Description: Put

# Data Cleaning

In [12]:
raw_file = RAW_DIR / "proteins_raw_10k.json"

with open(raw_file, "r") as f:
    proteins = json.load(f)

print(f"Loaded {len(proteins)} proteins from {raw_file}")

Loaded 10000 proteins from C:\Users\sridi\Documents\Learning\protein-function-active-learning\protein-function-active-learning\data\raw\proteins_raw_10k.json


## Data Preprocessing

In [13]:
def process_dataset(proteins):
    """
    Additional processing and quality control:
    - Valid amino acids only
    - Compute function word count
    """

    processed = []
    valid_aas = set("ACDEFGHIKLMNPQRSTVWY")

    skipped_reasons = {
        "invalid_aas": 0,
        "no_sequence": 0,
        "no_function": 0,
        "too_short_seq": 0,
        "too_short_func": 0,
    }

    for protein in tqdm(proteins, desc="Processing"):
        seq = protein.get("sequence", "").upper().strip()
        func = protein.get("function", "")

        # Must have sequence
        if not seq:
            skipped_reasons["no_sequence"] += 1
            continue

        # Must have function
        if not func:
            skipped_reasons["no_function"] += 1
            continue

        # Check for valid amino acids only
        if not all(aa in valid_aas for aa in seq):
            skipped_reasons["invalid_aas"] += 1
            continue

        # Minimum sequence length (very short sequences might be fragments)
        if len(seq) < 50:
            skipped_reasons["too_short_seq"] += 1
            continue

        # Minimum function description length
        if len(func) < 20:
            skipped_reasons["too_short_func"] += 1
            continue

        # Function length in words
        func_words = len(func.split())
        protein["function_words"] = func_words

        processed.append(protein)

    print(f"\nProcessed {len(processed)} proteins")
    print(f"Filtered out {len(proteins) - len(processed)} proteins:")
    for reason, count in skipped_reasons.items():
        if count > 0:
            print(f"   - {reason}: {count}")

    return processed

processed_proteins = process_dataset(proteins)

Processing:   0%|          | 0/10000 [00:00<?, ?it/s]


Processed 9987 proteins
Filtered out 13 proteins:
   - invalid_aas: 13


## Data Statistics

In [14]:
df_processed = pd.DataFrame(processed_proteins)

print(f"\n📊 Dataset Statistics:")
print(f"  - Total proteins: {len(processed_proteins)}")
print(f"  - Unique accession IDs: {df_processed['accession'].nunique()}")

print("\n📏 Sequence Length Distribution:")
print(df_processed["length"].describe())

print("\n📝 Function Word Count Distribution:")
print(df_processed["function_words"].describe())

print("\n🧪 Sample Protein:")
sample = df_processed.iloc[0]
print(f"  - Accession: {sample['accession']}")
print(f"  - Name: {sample['protein_name']}")
print(f"  - Gene: {sample['gene_name']}")
print(f"  - Organism: {sample['organism']}")
print(f"  - Sequence length: {sample['length']}")
print(f"  - Sequence: {sample['sequence'][:60]}...")
print(f"  - Function: {sample['function'][:200]}...")
print(f"  - Function words: {sample['function_words']}")
print(f"  - GO terms: {len(sample.get('go_terms', []))}")
print(f"  - EC numbers: {sample.get('ec_numbers', [])}")
print(f"  - Keywords: {sample.get('keywords', [])}")


📊 Dataset Statistics:
  - Total proteins: 9987
  - Unique accession IDs: 9987

📏 Sequence Length Distribution:
count     9987.000000
mean       624.342445
std        690.348363
min         51.000000
25%        291.500000
50%        463.000000
75%        745.000000
max      34350.000000
Name: length, dtype: float64

📝 Function Word Count Distribution:
count    9987.000000
mean       86.760288
std        89.649783
min         2.000000
25%        30.000000
50%        59.000000
75%       113.500000
max      1558.000000
Name: function_words, dtype: float64

🧪 Sample Protein:
  - Accession: A0A1B0GTW7
  - Name: Ciliated left-right organizer metallopeptidase
  - Gene: CIROP
  - Organism: Homo sapiens
  - Sequence length: 788
  - Sequence: MLLLLLLLLLLPPLVLRVAASRCLHDETQKSVSLLRPPFSQLPSKSRSSSLTLPSSRDPQ...
  - Function: Putative metalloproteinase that plays a role in left-right patterning process...
  - Function words: 10
  - GO terms: 8
  - EC numbers: ['3.4.24.-']
  - Keywords: [{'id': 'KW-0025

In [16]:
filtered_file = PROCESSED_DIR / "proteins_filtered_10k.json"

with open(filtered_file, "w") as f:
    json.dump(processed_proteins, f, indent=2)
    print(f"✓ Saved filtered data to {filtered_file}")

✓ Saved filtered data to C:\Users\sridi\Documents\Learning\protein-function-active-learning\protein-function-active-learning\data\processed\proteins_filtered_10k.json


# File Conversion - JSON to CSV

In [17]:
input_file = PROCESSED_DIR / "proteins_filtered_10k.json"
output_dir = PROCESSED_DIR

with open(input_file, "r", encoding="utf-8") as f:
    proteins = json.load(f)

main_rows = []
go_rows = []
keyword_rows = []

for protein in proteins:
    accession = protein.get("accession")
    go_terms = protein.get("go_terms", [])
    keywords = protein.get("keywords", [])

    main_rows.append({
        "accession": accession,
        "protein_name": protein.get("protein_name"),
        "gene_name": protein.get("gene_name"),
        "organism": protein.get("organism"),
        "sequence": protein.get("sequence"),
        "length": protein.get("length"),
        "function": protein.get("function"),
        "ec_numbers": ";".join(protein.get("ec_numbers", [])),
        "go_ids": ";".join([g.get("go_id", "") for g in go_terms]),
        "go_terms": ";".join([g.get("go_term", "") for g in go_terms]),
        "keywords": ";".join([k.get("name", "") for k in keywords]),
    })

    for go in go_terms:
        raw_term = go.get("go_term", "")

        if raw_term.startswith("F:"):
            namespace = "molecular_function"
            clean_term = raw_term[2:]
        elif raw_term.startswith("P:"):
            namespace = "biological_process"
            clean_term = raw_term[2:]
        elif raw_term.startswith("C:"):
            namespace = "cellular_component"
            clean_term = raw_term[2:]
        else:
            namespace = None
            clean_term = raw_term

        go_rows.append({
            "accession": accession,
            "go_id": go.get("go_id"),
            "go_term": clean_term,
            "go_namespace": namespace
        })

    for kw in keywords:
        keyword_rows.append({
            "accession": accession,
            "keyword_id": kw.get("id"),
            "keyword_name": kw.get("name"),
            "keyword_category": kw.get("category")
        })

pd.DataFrame(main_rows).to_csv(output_dir / "uniprot_proteins.csv", index=False)
pd.DataFrame(go_rows).to_csv(output_dir / "protein_go_terms.csv", index=False)
pd.DataFrame(keyword_rows).to_csv(output_dir / "protein_keywords.csv", index=False)

print("Saved:")
print(output_dir / "uniprot_proteins.csv")
print(output_dir / "protein_go_terms.csv")
print(output_dir / "protein_keywords.csv")

Saved:
C:\Users\sridi\Documents\Learning\protein-function-active-learning\protein-function-active-learning\data\processed\uniprot_proteins.csv
C:\Users\sridi\Documents\Learning\protein-function-active-learning\protein-function-active-learning\data\processed\protein_go_terms.csv
C:\Users\sridi\Documents\Learning\protein-function-active-learning\protein-function-active-learning\data\processed\protein_keywords.csv
